In [0]:
from pyspark.sql.types import DoubleType
from pyspark.sql import functions as F

dbutils.widgets.text("catalogo", "proyecto_ecommerce")
catalogo = dbutils.widgets.get("catalogo")

print(f"Catálogo: {catalogo}")

In [0]:
df_bronze_ordenes = spark.table(f"{catalogo}.bronze.ordenes")

total = df_bronze_ordenes.count()
fecha_epoch = df_bronze_ordenes.filter(F.col("fecha_hora").rlike("^[0-9]+$")).count()
precio_texto = df_bronze_ordenes.filter(~F.col("precio_unitario").rlike("^[0-9]+\\.?[0-9]*$")).count()
cantidad_negativa = df_bronze_ordenes.filter(F.col("cantidad") < 0).count()
cliente_nulo = df_bronze_ordenes.filter(F.col("cliente_id").isNull()).count()
orden_duplicada = total - df_bronze_ordenes.select("orden_id").distinct().count()

print(f"Total de filas: {total}")
print(f"fecha_hora en formato epoch: {fecha_epoch}")
print(f"precio_unitario como texto no-numérico limpio: {precio_texto}")
print(f"cantidad negativa: {cantidad_negativa}")
print(f"cliente_id nulo: {cliente_nulo}")
print(f"orden_id duplicados: {orden_duplicada}")

In [0]:

duplicados_ids_rows = (
    df_bronze_ordenes.groupBy("orden_id").count()
    .filter("count > 1")
    .limit(5)
    .select("orden_id")
    .collect()
)

duplicados_ids = [row["orden_id"] for row in duplicados_ids_rows]

display(df_bronze_ordenes.filter(F.col("orden_id").isin(duplicados_ids)).orderBy("orden_id"))

In [0]:
duplicados_agosto_rows = (
    df_bronze_ordenes
    .filter(F.col("orden_id").startswith("ORD202608"))
    .groupBy("orden_id").count()
    .filter("count > 1")
    .limit(5)
    .select("orden_id")
    .collect()
)

duplicados_agosto = [row["orden_id"] for row in duplicados_agosto_rows]
print(f"orden_id duplicados solo en lotes de agosto: {len(duplicados_agosto)}")

display(df_bronze_ordenes.filter(F.col("orden_id").isin(duplicados_agosto)).orderBy("orden_id"))

In [0]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalogo}.silver")

REGIONES_VALIDAS = ["Lima", "Arequipa", "Trujillo", "Cusco", "Piura", "Chiclayo"]

df_clientes_bronze = spark.table(f"{catalogo}.bronze.clientes")

df_clientes_silver = (
    df_clientes_bronze
    .dropDuplicates(["cliente_id"])
    .withColumn("nombre_cliente", F.initcap(F.trim("nombre_cliente")))
    .withColumn("region", F.initcap(F.trim("region")))
    .withColumn(
    "fecha_registro",
        F.coalesce(
            F.expr("try_to_date(fecha_registro, 'yyyy-MM-dd')"),
            F.expr("try_to_date(fecha_registro, 'dd/MM/yyyy')"),
            F.expr("try_to_date(fecha_registro, 'MM-dd-yyyy')"),
        ),
    )
    .filter(F.col("region").isin(REGIONES_VALIDAS))
    .select("cliente_id", "nombre_cliente", "email", "region", "segmento", "fecha_registro")
)

display(df_clientes_silver.limit(10))
print(f"Silver.clientes (aún sin guardar) -> {df_clientes_silver.count()} filas (de {df_clientes_bronze.count()} en bronze)")

In [0]:
(df_clientes_silver.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalogo}.silver.clientes"))

print(f"Guardado: {catalogo}.silver.clientes -> {df_clientes_silver.count()} filas")

In [0]:
df_productos_bronze = spark.table(f"{catalogo}.bronze.productos")

df_productos_silver = (
    df_productos_bronze
    .withColumn("precio_referencia", F.col("precio_referencia").cast(DoubleType()))
    .withColumn("categoria", F.initcap(F.trim("categoria")))
    .select("producto_id", "nombre_producto", "categoria", "marca", "precio_referencia")
)

display(df_productos_silver)
print(f"Silver.productos (aún sin guardar) -> {df_productos_silver.count()} filas")

In [0]:
(df_productos_silver.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalogo}.silver.productos"))

print(f"Guardado: {catalogo}.silver.productos -> {df_productos_silver.count()} filas")

In [0]:
df_ordenes_bronze = spark.table(f"{catalogo}.bronze.ordenes")

df_ordenes_ts = df_ordenes_bronze.withColumn(
    "fecha_hora_ts",
    F.when(
        F.col("fecha_hora").rlike("^[0-9]+$"),
        F.to_timestamp((F.col("fecha_hora").cast("double") / 1000)),
    ).otherwise(
        F.expr("try_to_timestamp(fecha_hora)")
    ),
)

display(
    df_ordenes_ts
    .select("orden_id", "fecha_hora", "fecha_hora_ts")
    .filter(F.col("fecha_hora").rlike("^[0-9]+$"))
    .limit(5)
)

In [0]:
df_ordenes_ts = (
    df_ordenes_ts
    .dropDuplicates(["orden_id"])
    .withColumn("precio_unitario", F.col("precio_unitario").cast(DoubleType()))
    .withColumn("canal", F.lower(F.trim("canal")))
)

total_tras_dedup = df_ordenes_ts.count()
print(f"Total tras deduplicar por orden_id: {total_tras_dedup} (de 4245 en bronze)")

In [0]:
##verificar que no tenga datos negativos o 0 en cantidad asi como no tenga null en cliente id
total_antes_filtro = df_ordenes_ts.count()

df_ordenes_validas = df_ordenes_ts.filter(
    (F.col("cantidad") > 0) & F.col("cliente_id").isNotNull()
)

descartadas = total_antes_filtro - df_ordenes_validas.count()

print(f"Total antes del filtro: {total_antes_filtro}")
print(f"Total después del filtro: {df_ordenes_validas.count()}")
print(f"Descartadas por cantidad inválida o cliente_id nulo: {descartadas}")


In [0]:
##filtrar tabla con las ordenes que lograron completarse
df_ordenes_silver = (
    df_ordenes_validas
    .filter(F.col("estado") == "completada")
    .withColumn("monto_bruto", F.round(F.col("cantidad") * F.col("precio_unitario"), 2))
    .withColumn(
        "monto_neto",
        F.round(F.col("monto_bruto") * (F.lit(1) - F.col("descuento_pct")), 2),
    )
    .withColumn("fecha", F.to_date("fecha_hora_ts"))
    .select(
        "orden_id", "fecha_hora_ts", "fecha", "cliente_id", "producto_id",
        "cantidad", "precio_unitario", "descuento_pct", "monto_bruto",
        "monto_neto", "canal", "estado",
    )
    .withColumnRenamed("fecha_hora_ts", "fecha_hora")
)

display(df_ordenes_silver.limit(10))

total_completadas = df_ordenes_silver.count()
descartadas_por_estado = df_ordenes_validas.count() - total_completadas
print(f"Órdenes completadas (silver final): {total_completadas}")
print(f"Descartadas por estado distinto de 'completada': {descartadas_por_estado}")

In [0]:
(df_ordenes_silver.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalogo}.silver.ordenes"))

print(f"Guardado: {catalogo}.silver.ordenes -> {df_ordenes_silver.count()} filas")